In [13]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0,
                      'C': 1,
                      'G': 2,
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [25]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [27]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [16]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [17]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [18]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [19]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [20]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [21]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides.

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .]
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .]
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [22]:
# Initiate two matrices:
# viterbi_value_matrix: to store the values described in the documentation above
# viterbi_trace_matrix: to store the path that led to the maximum value in each cell

num_states = len(states)           # 5 states: s, E, 5, I, e
seq_len = len(query_sequence)      # length of the observed sequence

# Initialize with -inf (log(0)) so impossible paths stay impossible
viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)
viterbi_trace_matrix = np.zeros((num_states, seq_len), dtype=int)

# --- First column (t=0): transition from start state 's' (index 0) ---
first_nuc = emission_nuc_codes[query_sequence[0]]
for s in range(num_states):
    trans_p = state_transition_prob[states['s']][s]
    emit_p  = emission_probs[s][first_nuc]
    if trans_p > 0 and emit_p > 0:
        viterbi_value_matrix[s][0] = math.log(trans_p) + math.log(emit_p)
    # trace for first column always points back to start state
    viterbi_trace_matrix[s][0] = states['s']

print('Viterbi value matrix (first column):')
print(viterbi_value_matrix[:, 0])

Viterbi value matrix (first column):
[       -inf -1.38629436        -inf        -inf        -inf]


### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND**

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [23]:
def calculate_prob_for_a_node(current_state, t):
    """
    For a given hidden state (row) and time step / column t,
    compute the best (max) log-probability arriving at this cell
    and the previous state that achieved it.

    Returns
    -------
    best_log_prob : float
        The maximum log probability for this cell.
    best_prev_state : int
        The index of the previous state that gave the max probability.
    """
    nuc = emission_nuc_codes[query_sequence[t]]
    emit_p = emission_probs[current_state][nuc]

    best_log_prob  = -np.inf
    best_prev_state = 0

    # If the emission probability is 0, this cell is impossible
    if emit_p == 0:
        return -np.inf, 0

    log_emit = math.log(emit_p)

    for prev_state in range(num_states):
        trans_p = state_transition_prob[prev_state][current_state]
        if trans_p == 0:
            continue
        prev_val = viterbi_value_matrix[prev_state][t - 1]
        if prev_val == -np.inf:
            continue

        candidate = prev_val + math.log(trans_p) + log_emit
        if candidate > best_log_prob:
            best_log_prob   = candidate
            best_prev_state = prev_state

    return best_log_prob, best_prev_state


# --- Fill the rest of the matrix (columns 1 .. seq_len-1) ---
for t in range(1, seq_len):
    for s in range(num_states):
        val, prev = calculate_prob_for_a_node(s, t)
        viterbi_value_matrix[s][t] = val
        viterbi_trace_matrix[s][t] = prev

print('Viterbi value matrix (last column):')
print(viterbi_value_matrix[:, -1])

Viterbi value matrix (last column):
[        -inf -38.67766628 -42.48432877 -38.91709259         -inf]


In [24]:
# Write a function to trace the state path that gave the maximum probability.
# This will be the final result.

def traceback():
    """
    Trace back through viterbi_trace_matrix to recover
    the most-probable hidden-state path.
    """
    # STEP 1: Find the maximum value in the last column of viterbi_value_matrix,
    #         because that is the one with the largest probability.
    #         The index of that value is the state of the last nucleotide.
    last_col = viterbi_value_matrix[:, -1]
    best_last_state = int(np.argmax(last_col))

    # STEP 2: Walk backwards through the trace matrix
    path_indices = [best_last_state]
    for t in range(seq_len - 1, 0, -1):
        prev = viterbi_trace_matrix[path_indices[-1]][t]
        path_indices.append(prev)

    path_indices.reverse()

    # STEP 3: Convert indices to state labels
    state_path = ''.join(id2state[idx] for idx in path_indices)
    return state_path


best_path = traceback()
print('Query:     ', query_sequence)
print('Best path: ', best_path)
print('Log prob:  ', viterbi_value_matrix[states[best_path[-1]], -1])

Query:      CTTCATGTGAAAGCAGACGTAAGTCA
Best path:  EEEEEEEEEEEEEEEEEEEEEEEEEE
Log prob:   -38.677666280562796
